In [1]:
!pip install timm --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
"""
Module 2 — Road Segmentation
Project: Automatic Road Network Extraction with Connectivity and Topology Refinement
Dataset: SpaceNet 5 (SN5) — AOI 8, Mumbai, India
Institution: VR Siddhartha Engineering College, Vijayawada

Architecture:
  - DeepLabV3+ with Xception backbone (pretrained on ImageNet)
  - ECA-Net (Efficient Channel Attention) replacing conventional SE block
  - ASPP (Atrous Spatial Pyramid Pooling) for multi-scale context
  - Binary Cross-Entropy loss
  - Adam optimizer
  - Metrics: IoU, F1-score
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    "output_dir":    "/home/jupyter-228w1a0529/Major/Dataset-1/processed",
    "checkpoint_dir": "/home/jupyter-228w1a0529/Major/checkpoints",

    "batch_size":    8,
    "num_workers":   4,
    "num_epochs":    50,
    "learning_rate": 1e-4,
    "threshold":     0.5,       # sigmoid threshold for binary prediction

    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


# ─────────────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────────────

class RoadDataset(Dataset):
    """Load patches from train.npz or val.npz produced by Module 1."""

    def __init__(self, npz_path: str):
        data = np.load(npz_path)
        self.images = data["images"]   # (N, 3, H, W) float32
        self.masks  = data["masks"]    # (N, H, W)    float32
        print(f"[INFO] Loaded {len(self.images)} patches from {Path(npz_path).name}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.images[idx]),
            torch.from_numpy(self.masks[idx]).unsqueeze(0)   # (1, H, W)
        )


def get_dataloaders(config=CONFIG):
    out = Path(config["output_dir"])
    train_ds = RoadDataset(out / "train.npz")
    val_ds   = RoadDataset(out / "val.npz")
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"],
                              shuffle=True,  num_workers=config["num_workers"],
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=config["batch_size"],
                              shuffle=False, num_workers=config["num_workers"],
                              pin_memory=True)
    print(f"[INFO] Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────────────────
# ECA-NET — Efficient Channel Attention
# ─────────────────────────────────────────────────────────────────────────────

class ECABlock(nn.Module):
    """
    Efficient Channel Attention (ECA-Net, Wang et al. 2020).

    How it works:
      1. Global average pool: squeeze spatial dims → (B, C, 1, 1)
      2. 1D conv across channels with small adaptive kernel
         (avoids dimensionality reduction used in SE block)
      3. Sigmoid → channel-wise attention weights
      4. Multiply input feature map by weights (recalibrate channels)

    Advantage over SE block: no FC layers, fewer parameters,
    better gradient flow, maintains channel interdependencies.
    """

    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        # Adaptive kernel size: odd number close to log2(C)
        import math
        t = int(abs(math.log2(channels) / gamma + b / gamma))
        k = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv1d   = nn.Conv1d(1, 1, kernel_size=k,
                                  padding=k // 2, bias=False)
        self.sigmoid  = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W)
        y = self.avg_pool(x)          # (B, C, 1, 1)
        y = y.squeeze(-1)             # (B, C, 1)
        y = y.transpose(-1, -2)       # (B, 1, C)
        y = self.conv1d(y)            # (B, 1, C)
        y = y.transpose(-1, -2)       # (B, C, 1)
        y = y.unsqueeze(-1)           # (B, C, 1, 1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)    # recalibrate channels


# ─────────────────────────────────────────────────────────────────────────────
# ASPP — Atrous Spatial Pyramid Pooling
# ─────────────────────────────────────────────────────────────────────────────

class ASPPConv(nn.Sequential):
    """Single ASPP branch: dilated conv → BN → ReLU."""
    def __init__(self, in_ch, out_ch, dilation):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation,
                      dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )


class ASPPPooling(nn.Sequential):
    """Global average pooling branch of ASPP — uses GroupNorm instead of BatchNorm."""
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(32, out_ch),   # ← replaces BatchNorm2d (works on 1×1)
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[-2:]
        for mod in self:
            x = mod(x)
        return F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        
class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling.

    Runs 5 parallel branches at different dilation rates to capture
    multi-scale context — critical for detecting roads of varying widths.

    Branches:
      1. 1×1 conv (rate=1)   — fine local detail
      2. 3×3 conv (rate=6)   — small context
      3. 3×3 conv (rate=12)  — medium context
      4. 3×3 conv (rate=18)  — large context
      5. Global avg pool     — image-level context

    All branches concatenated → 1×1 conv to fuse → ECA attention
    """

    def __init__(self, in_ch: int, out_ch: int = 256):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        self.branch2  = ASPPConv(in_ch, out_ch, dilation=6)
        self.branch3  = ASPPConv(in_ch, out_ch, dilation=12)
        self.branch4  = ASPPConv(in_ch, out_ch, dilation=18)
        self.branch5  = ASPPPooling(in_ch, out_ch)

        self.project  = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )
        self.eca = ECABlock(out_ch)   # ECA after ASPP fusion

    def forward(self, x):
        branches = [
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x),
            self.branch5(x),
        ]
        x = torch.cat(branches, dim=1)
        x = self.project(x)
        x = self.eca(x)               # channel attention after fusion
        return x


# ─────────────────────────────────────────────────────────────────────────────
# DEEPLABV3+ WITH XCEPTION BACKBONE
# ─────────────────────────────────────────────────────────────────────────────

class DeepLabV3Plus(nn.Module):
    """
    DeepLabV3+ with Xception backbone + ECA-Net attention.

    Architecture:
      Encoder (Xception):
        - Pretrained Xception extracts deep features
        - output_stride=16: balances resolution vs receptive field
        - low_level features from early layers for decoder skip connection

      ASPP:
        - Multi-scale context aggregation on encoder output
        - ECA channel attention after ASPP fusion

      Decoder:
        - Upsample ASPP output × 4
        - Concatenate with low-level features (after 1×1 conv)
        - Two 3×3 convs to refine
        - ECA attention on combined features
        - Upsample × 4 → full resolution
        - 1×1 conv → single-channel logit

    Input:  (B, 3, 512, 512)
    Output: (B, 1, 512, 512)  — raw logits, apply sigmoid for probability
    """

    def __init__(self, num_classes: int = 1):
        super().__init__()

        # ── Backbone: Xception (pretrained) ──────────────────────────────────
        # We use torchvision's feature_extraction or timm for Xception.
        # timm gives cleaner access to intermediate features.
        try:
            import timm
            self.backbone = timm.create_model(
                "xception",
                pretrained=True,
                features_only=True,
                out_indices=(1, 3)   # low-level (idx 1) and high-level (idx 3)
            )
            # Get channel counts from backbone
            dummy = torch.zeros(1, 3, 512, 512)
            with torch.no_grad():
                feats = self.backbone(dummy)
            low_ch  = feats[0].shape[1]   # low-level feature channels
            high_ch = feats[1].shape[1]   # high-level feature channels
            print(f"[INFO] Backbone: Xception | low_ch={low_ch} | high_ch={high_ch}")

        except ImportError:
            raise ImportError(
                "timm not found. Install with: pip install timm --break-system-packages"
            )

        # ── ASPP ─────────────────────────────────────────────────────────────
        self.aspp = ASPP(in_ch=high_ch, out_ch=256)

        # ── Low-level feature projection (1×1 conv to reduce channels) ───────
        self.low_level_proj = nn.Sequential(
            nn.Conv2d(low_ch, 48, 1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )

        # ── Decoder ───────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.decoder_eca = ECABlock(256)   # ECA in decoder

        # ── Final classification head ─────────────────────────────────────────
        self.classifier = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        input_size = x.shape[-2:]

        # Encoder
        low_feat, high_feat = self.backbone(x)   # low: (B,128,H/4,W/4) high: (B,2048,H/16,W/16)

        # ASPP on high-level features
        x = self.aspp(high_feat)                 # (B, 256, H/16, W/16)

        # Upsample ASPP output to match low-level feature size
        x = F.interpolate(x, size=low_feat.shape[-2:],
                          mode="bilinear", align_corners=False)

        # Project low-level features and concatenate
        low_feat = self.low_level_proj(low_feat)   # (B, 48, H/4, W/4)
        x = torch.cat([x, low_feat], dim=1)        # (B, 304, H/4, W/4)

        # Decode
        x = self.decoder(x)                        # (B, 256, H/4, W/4)
        x = self.decoder_eca(x)                    # ECA attention

        # Upsample to original resolution
        x = F.interpolate(x, size=input_size,
                          mode="bilinear", align_corners=False)

        # Final logit (no sigmoid here — applied in loss / prediction)
        x = self.classifier(x)                     # (B, 1, H, W)
        return x


# ─────────────────────────────────────────────────────────────────────────────
# LOSS FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

class BCEDiceLoss(nn.Module):
    """
    Combined Binary Cross-Entropy + Dice Loss.

    BCE alone struggles with class imbalance (few road pixels vs many background).
    Dice loss directly optimizes the F1-score metric we care about.
    Equal weighting (0.5 each) works well for road segmentation.
    """

    def __init__(self, bce_weight: float = 0.5):
        super().__init__()
        self.bce_weight  = bce_weight
        self.dice_weight = 1.0 - bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # BCE loss
        bce_loss = self.bce(logits, targets)

        # Dice loss
        probs  = torch.sigmoid(logits)
        smooth = 1.0
        intersection = (probs * targets).sum(dim=(1, 2, 3))
        dice_loss = 1 - (2 * intersection + smooth) / \
                    (probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + smooth)
        dice_loss = dice_loss.mean()

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────

def compute_metrics(logits: torch.Tensor,
                    targets: torch.Tensor,
                    threshold: float = 0.5):
    """
    Compute IoU and F1-score from raw logits and binary targets.

    IoU  = TP / (TP + FP + FN)
    F1   = 2*TP / (2*TP + FP + FN)
    """
    preds = (torch.sigmoid(logits) > threshold).float()

    TP = (preds * targets).sum(dim=(1, 2, 3))
    FP = (preds * (1 - targets)).sum(dim=(1, 2, 3))
    FN = ((1 - preds) * targets).sum(dim=(1, 2, 3))

    smooth = 1e-6
    iou = (TP + smooth) / (TP + FP + FN + smooth)
    f1  = (2 * TP + smooth) / (2 * TP + FP + FN + smooth)

    return iou.mean().item(), f1.mean().item()


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train(config=CONFIG):
    device = torch.device(config["device"])
    print(f"[INFO] Using device: {device}")

    # Directories
    ckpt_dir = Path(config["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # Data
    train_loader, val_loader = get_dataloaders(config)

    # Model
    model = DeepLabV3Plus(num_classes=1).to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] Trainable parameters: {total_params:,}")

    # Loss, optimizer, scheduler
    criterion = BCEDiceLoss(bce_weight=0.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5, verbose=True
    )

    # Training history
    history = {"train_loss": [], "val_loss": [],
               "train_iou":  [], "val_iou":  [],
               "train_f1":   [], "val_f1":   []}

    best_val_iou = 0.0

    for epoch in range(1, config["num_epochs"] + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_loss = train_iou = train_f1 = 0.0

        for images, masks in tqdm(train_loader,
                                  desc=f"Epoch {epoch}/{config['num_epochs']} [Train]",
                                  leave=False):
            images = images.to(device)
            masks  = masks.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss   = criterion(logits, masks)
            loss.backward()
            optimizer.step()

            iou, f1 = compute_metrics(logits.detach(), masks, config["threshold"])
            train_loss += loss.item()
            train_iou  += iou
            train_f1   += f1

        n = len(train_loader)
        train_loss /= n;  train_iou /= n;  train_f1 /= n

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        val_loss = val_iou = val_f1 = 0.0

        with torch.no_grad():
            for images, masks in tqdm(val_loader,
                                      desc=f"Epoch {epoch}/{config['num_epochs']} [Val]",
                                      leave=False):
                images = images.to(device)
                masks  = masks.to(device)
                logits = model(images)
                loss   = criterion(logits, masks)
                iou, f1 = compute_metrics(logits, masks, config["threshold"])
                val_loss += loss.item()
                val_iou  += iou
                val_f1   += f1

        n = len(val_loader)
        val_loss /= n;  val_iou /= n;  val_f1 /= n

        # ── Log ───────────────────────────────────────────────────────────────
        print(f"Epoch {epoch:03d} | "
              f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
              f"IoU: {train_iou:.4f}/{val_iou:.4f} | "
              f"F1: {train_f1:.4f}/{val_f1:.4f}")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_iou"].append(train_iou)
        history["val_iou"].append(val_iou)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        # ── Scheduler step (based on val IoU) ─────────────────────────────────
        scheduler.step(val_iou)

        # ── Save best checkpoint ───────────────────────────────────────────────
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save({
                "epoch":      epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_iou":    val_iou,
                "val_f1":     val_f1,
            }, ckpt_dir / "best_model.pth")
            print(f"  ✅ Saved best model (val IoU={val_iou:.4f})")

        # ── Save latest checkpoint every 10 epochs ─────────────────────────────
        if epoch % 10 == 0:
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
            }, ckpt_dir / f"checkpoint_epoch{epoch:03d}.pth")

    # ── Plot training curves ───────────────────────────────────────────────────
    plot_history(history)
    print(f"\n[INFO] ✅ Training complete. Best val IoU: {best_val_iou:.4f}")
    return model, history


# ─────────────────────────────────────────────────────────────────────────────
# PLOT TRAINING HISTORY
# ─────────────────────────────────────────────────────────────────────────────

def plot_history(history: dict):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

    axes[1].plot(history["train_iou"], label="Train")
    axes[1].plot(history["val_iou"],   label="Val")
    axes[1].set_title("IoU"); axes[1].legend(); axes[1].set_xlabel("Epoch")

    axes[2].plot(history["train_f1"], label="Train")
    axes[2].plot(history["val_f1"],   label="Val")
    axes[2].set_title("F1-score"); axes[2].legend(); axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()
    print("[INFO] Training curves saved to training_curves.png")


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE — PREDICT ON A SINGLE IMAGE PATCH
# ─────────────────────────────────────────────────────────────────────────────

def predict(model, image_tensor: torch.Tensor,
            threshold: float = 0.5,
            device: str = "cuda") -> np.ndarray:
    """
    Run inference on a single normalized image patch.

    Parameters
    ----------
    model        : trained DeepLabV3Plus model
    image_tensor : (3, 512, 512) float32 tensor (already normalized)
    threshold    : sigmoid threshold for binary mask

    Returns
    -------
    mask : (512, 512) uint8 numpy array — 1=road, 0=background
    """
    model.eval()
    with torch.no_grad():
        x = image_tensor.unsqueeze(0).to(device)   # (1, 3, 512, 512)
        logit = model(x)                            # (1, 1, 512, 512)
        prob  = torch.sigmoid(logit).squeeze().cpu().numpy()
    return (prob > threshold).astype(np.uint8)


def visualize_predictions(model, val_npz: str, n: int = 4,
                           threshold: float = 0.5, device: str = "cuda"):
    """Show n sample predictions from the validation set."""
    data   = np.load(val_npz)
    images = data["images"]
    masks  = data["masks"]

    idxs = np.random.choice(len(images), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    titles = ["Input image", "Ground truth", "Prediction"]

    for i, idx in enumerate(idxs):
        img_tensor = torch.from_numpy(images[idx])
        pred_mask  = predict(model, img_tensor, threshold, device)

        # Denormalize for display
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_display = np.transpose(images[idx], (1, 2, 0))
        img_display = (img_display * std + mean).clip(0, 1)

        axes[i][0].imshow(img_display);         axes[i][0].set_title(titles[0]); axes[i][0].axis("off")
        axes[i][1].imshow(masks[idx], cmap="gray"); axes[i][1].set_title(titles[1]); axes[i][1].axis("off")
        axes[i][2].imshow(pred_mask,  cmap="gray"); axes[i][2].set_title(titles[2]); axes[i][2].axis("off")

    plt.tight_layout()
    plt.savefig("sample_predictions.png", dpi=150)
    plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Install timm if needed:
    # pip install timm --break-system-packages

    print(f"[INFO] Device: {CONFIG['device']}")
    print(f"[INFO] PyTorch: {torch.__version__}")

    # Train
    model, history = train(CONFIG)

    # Visualize predictions on validation set
    val_npz = Path(CONFIG["output_dir"]) / "val.npz"
    visualize_predictions(model, str(val_npz),
                          n=4,
                          threshold=CONFIG["threshold"],
                          device=CONFIG["device"])

[INFO] Device: cuda
[INFO] PyTorch: 2.5.1+cu121
[INFO] Using device: cuda


KeyboardInterrupt: 